# 22.12 边缘部署:模型压缩三件套 / Edge Deployment: Model Compression

**中文**:前面的部署都假设模型跑在**服务器**上(有 GPU、有内存、有网络)。但很多场景要把模型放到**边缘设备**:手机(相册分类、语音识别)、IoT 传感器、浏览器、汽车、摄像头。这些设备**内存小、算力弱、可能没网络、要求低延迟、还常涉及隐私**(数据不上传)。问题是:一个几百 MB、需要 GPU 的模型,根本塞不进手机、也跑不动。解法是**模型压缩(model compression)**:在尽量不掉精度的前提下,把模型变小变快。本节用真实的 PyTorch 实验,亲手做**压缩三件套——量化(quantization)、剪枝(pruning)、知识蒸馏(distillation)**,并诚实测量每种方法的"体积/精度"权衡。这是把 AI 送到十亿台设备上的关键工程。
**English**: Prior deployment assumed the model runs on a **server** (with GPU, memory, network). But many scenarios put models on **edge devices**: phones (photo classification, speech recognition), IoT sensors, browsers, cars, cameras. These have **small memory, weak compute, possibly no network, low-latency requirements, and often privacy concerns** (data not uploaded). The problem: a several-hundred-MB, GPU-requiring model simply won't fit on a phone or run on it. The solution is **model compression**: shrink and speed up the model while minimizing accuracy loss. This section uses real PyTorch experiments to hand-do the **compression trio — quantization, pruning, knowledge distillation** — and honestly measures each method's "size/accuracy" tradeoff. This is the key engineering to bring AI to billions of devices.

---

**中文**:**模型压缩三件套**:
**English**: **The model compression trio**:
- **中文**:**量化(quantization)**:把权重从 32 位浮点(fp32)降到 8 位整数(int8),甚至更低。**每个数从 4 字节变 1 字节 → 模型直接小 4 倍、且整数运算更快**。神奇的是,神经网络对精度损失很鲁棒,int8 量化通常**几乎不掉精度**——这是性价比最高的压缩。
  **Quantization**: reduce weights from 32-bit float (fp32) to 8-bit integer (int8) or lower. **Each number goes from 4 bytes to 1 byte → the model is 4x smaller, and integer math is faster**. Remarkably, neural nets are robust to precision loss, and int8 quantization usually **barely drops accuracy** — the highest bang-for-buck compression.
- **中文**:**剪枝(pruning)**:把**绝对值很小、贡献很低的权重直接置零**(网络里大量权重接近 0、几乎没用)。得到稀疏模型,用稀疏存储/推理能省内存和算力。
  **Pruning**: **zero out weights with tiny absolute values and low contribution** (many weights in a net are near 0 and nearly useless). Yields a sparse model, and sparse storage/inference saves memory and compute.
- **中文**:**知识蒸馏(knowledge distillation)**:训一个**小"学生"模型去模仿大"老师"模型的输出**——不只学硬标签(是/不是猫),还学老师的**软标签**(0.7猫/0.2狗/0.1狐狸,这些"类别间的相似度"叫暗知识 dark knowledge)。让小模型"继承"大模型的知识。DistilBERT 就是 BERT 蒸馏来的。
  **Knowledge distillation**: train a small "student" model to **mimic a large "teacher" model's outputs** — learning not just hard labels (is/isn't a cat) but the teacher's **soft labels** (0.7 cat / 0.2 dog / 0.1 fox — these "inter-class similarities" are called dark knowledge). The small model "inherits" the big model's knowledge. DistilBERT is distilled from BERT.

> 💡 **面试速查 / Interview cheat-sheet（★★ 边缘/移动 AI 必考）**
> **中文**:**边缘部署**需求:小内存/弱算力/低延迟/离线/隐私→必须**压缩模型**。**三件套**:①**量化**(fp32→int8/更低, 4x 小+整数运算快, **几乎不掉精度=最高性价比**; PTQ 训练后量化最简单, QAT 量化感知训练精度更好)②**剪枝**(置零小权重, 结构化剪枝=删整个通道/头对硬件更友好, 非结构化=单权重稀疏需专门支持)③**知识蒸馏**(小学生模仿大老师的软标签/暗知识, 如 DistilBERT/TinyBERT)。还有:**低秩分解**、**权重共享**、**NAS 搜小模型**。**部署格式/运行时**:**ONNX Runtime**、**TFLite**(移动)、**Core ML**(iOS)、**TensorRT**(NVIDIA)、**GGUF/llama.cpp**(端侧 LLM), 都做算子融合+量化加速。**权衡**:体积/延迟/功耗 vs 精度——按设备约束选。**LLM 端侧**:4-bit 量化(GPTQ/AWQ)是关键。面试金句:*"边缘设备内存算力有限, 要压缩模型:量化(fp32→int8, 4x小几乎不掉精度, 性价比最高)、剪枝(置零小权重, 结构化更硬件友好)、知识蒸馏(小模型学大模型软标签); 部署用 TFLite/ONNX Runtime/Core ML/TensorRT 做量化和算子融合; 端侧 LLM 靠 4-bit 量化(GPTQ/AWQ); 核心是体积/延迟/功耗与精度的权衡。"*
> **English**: **Edge deployment** needs: small memory/weak compute/low latency/offline/privacy → must **compress the model**. **The trio**: ① **quantization** (fp32→int8/lower, 4x smaller + fast integer math, **barely drops accuracy = best bang-for-buck**; PTQ post-training quantization is simplest, QAT quantization-aware training is more accurate) ② **pruning** (zero small weights; structured pruning = remove whole channels/heads, more hardware-friendly; unstructured = per-weight sparsity, needs special support) ③ **knowledge distillation** (a small student mimics a big teacher's soft labels/dark knowledge, e.g. DistilBERT/TinyBERT). Also: **low-rank factorization**, **weight sharing**, **NAS for small models**. **Deployment formats/runtimes**: **ONNX Runtime**, **TFLite** (mobile), **Core ML** (iOS), **TensorRT** (NVIDIA), **GGUF/llama.cpp** (on-device LLMs), all doing operator fusion + quantization acceleration. **Tradeoff**: size/latency/power vs accuracy — choose by device constraints. **On-device LLMs**: 4-bit quantization (GPTQ/AWQ) is key. Interview line: *"Edge devices have limited memory/compute, so compress models: quantization (fp32→int8, 4x smaller, barely drops accuracy, best value), pruning (zero small weights, structured is more hardware-friendly), knowledge distillation (small model learns the big model's soft labels); deploy with TFLite/ONNX Runtime/Core ML/TensorRT for quantization and operator fusion; on-device LLMs rely on 4-bit quantization (GPTQ/AWQ); the core is trading size/latency/power against accuracy."*


In [ ]:

# ============================================================
# 训练一个"老师"模型 + 量化 + 剪枝(真实 PyTorch 测量)/ train a teacher + quantize + prune (real measurement)
# 中文:在 MNIST 上训一个较大的 MLP 作老师, 然后压缩它, 测量"体积 vs 精度"的真实权衡。
# English: train a larger MLP teacher on MNIST, then compress it, measuring the real "size vs accuracy" tradeoff.
# ============================================================
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np, os, copy, warnings
warnings.filterwarnings("ignore")
torch.manual_seed(0); np.random.seed(0)
from torchvision import datasets
root=os.path.expanduser("~/.cache/dsfs_cv")
mn=datasets.MNIST(root, train=True, download=False)
idx=np.random.permutation(len(mn.data))[:12000]
X=(mn.data.float()/255.).view(-1,784)[idx]; y=mn.targets[idx]
te=datasets.MNIST(root, train=False, download=False)
Xte=(te.data.float()/255.).view(-1,784)[:2000]; yte=te.targets[:2000]
def accuracy(m):
    m.eval()
    with torch.no_grad(): return (m(Xte).argmax(1)==yte).float().mean().item()
def train(m, ep=6, lr=0.1):
    opt=torch.optim.SGD(m.parameters(), lr)
    for _ in range(ep):
        for b in range(0,len(X),128): opt.zero_grad(); F.cross_entropy(m(X[b:b+128]),y[b:b+128]).backward(); opt.step()
    return m
nparams=lambda m: sum(p.numel() for p in m.parameters())

teacher=nn.Sequential(nn.Linear(784,256),nn.ReLU(),nn.Linear(256,256),nn.ReLU(),nn.Linear(256,10))
train(teacher); ta=accuracy(teacher); tp=nparams(teacher)
print(f"老师模型 Teacher (fp32):  精度 {ta:.3f}   参数 {tp:,}   体积 {tp*4/1024:.0f}KB  (fp32=4字节/参数)")

# ① 量化:从零把权重量化成 int8(对称量化: scale=max|w|/127)再反量化, 测精度 / from-scratch int8 quantization
quant=copy.deepcopy(teacher)
with torch.no_grad():
    for p in quant.parameters():
        scale=p.abs().max()/127                            # 量化步长 / quantization scale
        p.copy_((p/scale).round().clamp(-127,127)*scale)   # 量化→反量化(模拟 int8 存储)/ quantize→dequantize
print(f"① 量化 int8:            精度 {accuracy(quant):.3f}   体积 {tp*1/1024:.0f}KB   → 小 4x, 精度几乎不变!")

# ② 剪枝:把最小的 80% 权重置零 / prune the smallest 80% of weights to zero
pruned=copy.deepcopy(teacher)
with torch.no_grad():
    for p in pruned.parameters():
        if p.dim()==2:
            thr=p.abs().flatten().kthvalue(int(0.8*p.numel())).values; p[p.abs()<thr]=0   # 阈值以下置零 / zero below threshold
tot=sum(p.numel() for p in pruned.parameters() if p.dim()==2)
zeros=sum((p==0).sum().item() for p in pruned.parameters() if p.dim()==2)
print(f"② 剪枝 80%:             精度 {accuracy(pruned):.3f}   稀疏度 {zeros/tot:.0%}   → 稀疏存储可省数倍内存")


In [ ]:

# ============================================================
# ③ 知识蒸馏:小学生模仿大老师 / knowledge distillation: a small student mimics the big teacher
# 中文:训一个参数少一个数量级的"学生"。对比:(a)直接用硬标签从零训; (b)用老师的软标签蒸馏。
# English: train a "student" with an order of magnitude fewer parameters. Compare: (a) from-scratch on hard labels;
#      (b) distilled from the teacher's soft labels.
# ============================================================
def make_student(): return nn.Sequential(nn.Linear(784,32),nn.ReLU(),nn.Linear(32,10))   # 小学生 / tiny student
sp=nparams(make_student())
# (a) 学生从零训(硬标签)/ student from scratch (hard labels)
torch.manual_seed(1); s_scratch=train(make_student())
# (b) 蒸馏:学生学老师的软标签(暗知识)+ 硬标签 / distill: soft labels (dark knowledge) + hard labels
torch.manual_seed(1); s_distill=make_student(); opt=torch.optim.SGD(s_distill.parameters(),0.1); T=4.0; teacher.eval()
for _ in range(6):
    for b in range(0,len(X),128):
        with torch.no_grad(): soft=F.softmax(teacher(X[b:b+128])/T, dim=1)               # 老师的软标签 / teacher soft labels
        opt.zero_grad()
        kd=F.kl_div(F.log_softmax(s_distill(X[b:b+128])/T,dim=1), soft, reduction="batchmean")*T*T  # 蒸馏损失 / KD loss
        ce=F.cross_entropy(s_distill(X[b:b+128]), y[b:b+128])                             # 硬标签损失 / hard-label loss
        (0.5*kd+0.5*ce).backward(); opt.step()
print(f"③ 知识蒸馏 / distillation:")
print(f"   老师(大): 参数 {tp:,}, 精度 {ta:.3f}")
print(f"   学生从零(硬标签): 参数 {sp:,} ({tp/sp:.0f}x 更少), 精度 {accuracy(s_scratch):.3f}")
print(f"   学生蒸馏(软标签): 参数 {sp:,},              精度 {accuracy(s_distill):.3f}")
print(f"   → 学生只用老师 1/{tp//sp} 的参数(体积 {sp*4/1024:.0f}KB), 精度接近老师——小模型足以上手机")
print("   ⚠️诚实:本例任务简单, 小学生用硬标签已学得很好, 蒸馏未必更优(见下方诚实解读)")


In [ ]:

# ============================================================
# 可视化:压缩三件套的体积/精度权衡 / size vs accuracy tradeoff of the trio
# ============================================================
import matplotlib.pyplot as plt
methods=["老师\n(fp32)","量化\n(int8)","剪枝\n(80%)","小学生\n(蒸馏)"]
sizes=[tp*4/1024, tp*1/1024, tp*4*0.2/1024, sp*4/1024]     # KB(剪枝按稀疏存储估、蒸馏按学生参数)/ approx sizes
accs=[ta, accuracy(quant), accuracy(pruned), accuracy(s_distill)]
fig,ax=plt.subplots(1,2,figsize=(14,5))
b=ax[0].bar(methods,sizes,color=["#4C72B0","#55A868","#DD8452","#9467BD"])
for bar,s in zip(b,sizes): ax[0].text(bar.get_x()+bar.get_width()/2,s+10,f"{s:.0f}KB",ha="center",fontsize=10,weight="bold")
ax[0].set_ylabel("模型体积 KB(越小越好)"); ax[0].set_title("压缩三件套:体积大幅下降")
ax[1].bar(methods,accs,color=["#4C72B0","#55A868","#DD8452","#9467BD"])
for i,a in enumerate(accs): ax[1].text(i,a+0.005,f"{a:.3f}",ha="center",fontsize=10,weight="bold")
ax[1].set_ylim(0.7,0.95); ax[1].axhline(ta,ls="--",color="gray",alpha=0.6,label="老师精度")
ax[1].set_ylabel("测试精度"); ax[1].set_title("精度基本保持(量化几乎无损)"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig("/tmp/mlops12_viz.png",dpi=80); plt.show()
print(f"量化: {tp*4/1024:.0f}KB→{tp*1/1024:.0f}KB(4x小)几乎无损; 剪枝: 80%稀疏小掉点; 学生: {tp//sp}x少参数上手机")


**中文**:诚实解读:
**English**: Honest takeaways:

**中文**:
1. **量化是"免费的午餐",压缩里性价比最高的一招**:我们的实验一目了然——把权重从 32 位浮点降到 8 位整数,模型**直接小 4 倍(1052KB→263KB),精度却几乎纹丝不动(0.886→0.886)**。这看似违反直觉(降精度不该掉性能吗?),但神经网络对权重的数值精度出奇地鲁棒——因为它靠的是大量权重的**协同**,而非单个权重的精确值。所以量化几乎是**无脑就该做**的第一步压缩:体积小 4 倍、整数运算更快、功耗更低,代价却微乎其微。这也是为什么所有移动端/边缘推理框架(TFLite、Core ML、ONNX Runtime)都把量化作为核心能力,以及为什么端侧大模型(手机上跑 LLM)几乎都靠 4-bit 量化。
2. **剪枝和"选对小架构"从另外两个角度压缩**:①**剪枝**利用了"网络里大量权重接近 0、几乎没用"这个事实——把最小的 80% 权重置零,精度只从 0.886 掉到 0.871,却得到 80% 稀疏的模型。但要真正省内存/算力,需要**硬件/框架支持稀疏运算**(非结构化剪枝的稀疏往往难加速),所以实践中更常用**结构化剪枝**(删掉整个通道/注意力头),对硬件更友好。②**直接用一个小架构**(我们的学生只有老师 1/11 的参数、99KB)在这个任务上精度几乎不输老师(0.883 vs 0.886)——提醒我们:**有时最好的"压缩"不是压大模型,而是一开始就用合适大小的模型**。杀鸡不用牛刀。
3. **诚实的意外:知识蒸馏在这个简单任务上没赢过"从零训小模型"**。理论上,蒸馏让小学生学习老师的**软标签**(暗知识:"这个 7 有点像 1"),应该比只学硬标签更好。但我们的实验里,蒸馏学生(0.864)反而略低于从零训练的学生(0.883)。**这不是 bug,而是一个重要的诚实教训**:①**任务太简单时蒸馏没用武之地**——MNIST 上一个 32 隐层的小模型用硬标签就能学到 0.88,已经逼近老师的 0.886,几乎没有"老师会而学生学不会"的知识可传递,软标签的额外信息量微乎其微。②**蒸馏的真正价值在"学生靠自己学不好"的场景**:当任务很难、学生容量相对不足、或数据有限时,老师的软标签才提供显著更丰富的监督信号——这正是 **DistilBERT(BERT 蒸馏,保留 97% 性能、小 40%、快 60%)** 成功的地方(语言理解很难,小模型从硬标签学不到那么好)。③**蒸馏对超参敏感**(温度 T、软硬标签权重、训练时长),需要调。**结论:边缘部署靠压缩三件套把模型送上设备——量化几乎是免费的 4x 压缩(首选)、剪枝用稀疏性换体积(结构化更实用)、蒸馏把大模型知识传给小模型(在难任务上才显著有效);但诚实地说, 压缩不是越复杂越好——量化和'选对小架构'常常是最简单也最有效的招, 复杂的蒸馏只在'小模型自己学不好'时才兑现价值。选哪种, 取决于你的设备约束和任务难度。**

**English**:
1. **Quantization is a "free lunch," the highest-value compression move**: our experiment is unmistakable — reducing weights from 32-bit float to 8-bit integer makes the model **4x smaller (1052KB→263KB) while accuracy barely moves (0.886→0.886)**. This seems counterintuitive (shouldn't lower precision hurt?), but neural nets are surprisingly robust to weight numerical precision — because they rely on the **coordination** of many weights, not any single weight's exact value. So quantization is almost a **no-brainer** first compression step: 4x smaller, faster integer math, lower power, at negligible cost. This is why all mobile/edge inference frameworks (TFLite, Core ML, ONNX Runtime) make quantization a core capability, and why on-device large models (LLMs on phones) almost all rely on 4-bit quantization.
2. **Pruning and "choosing the right small architecture" compress from two other angles**: ① **Pruning** exploits the fact that "many weights in a net are near 0 and nearly useless" — zeroing the smallest 80% drops accuracy only from 0.886 to 0.871 while yielding an 80%-sparse model. But to truly save memory/compute, you need **hardware/framework support for sparse ops** (unstructured sparsity is often hard to accelerate), so in practice **structured pruning** (removing whole channels/attention heads) is more common, being more hardware-friendly. ② **Just using a small architecture** (our student has 1/11 the teacher's parameters, 99KB) nearly matches the teacher on this task (0.883 vs 0.886) — reminding us that **sometimes the best "compression" isn't compressing a big model but starting with a right-sized one**. Don't use a sledgehammer to crack a nut.
3. **Honest surprise: knowledge distillation didn't beat "training a small model from scratch" on this easy task**. In theory, distillation lets the small student learn the teacher's **soft labels** (dark knowledge: "this 7 looks a bit like a 1"), which should beat hard labels alone. But in our experiment, the distilled student (0.864) was slightly below the from-scratch student (0.883). **This isn't a bug but an important honest lesson**: ① **When the task is too easy, distillation has no room** — on MNIST a 32-hidden small model learns 0.88 from hard labels alone, already near the teacher's 0.886, with almost no "knowledge the teacher has but the student can't learn" to transfer, so soft labels add negligible information. ② **Distillation's real value is where "the student can't learn well on its own"**: when the task is hard, the student's capacity is relatively insufficient, or data is limited, the teacher's soft labels provide substantially richer supervision — exactly where **DistilBERT (distilled from BERT, retaining 97% performance, 40% smaller, 60% faster)** succeeds (language understanding is hard, a small model can't learn as well from hard labels). ③ **Distillation is hyperparameter-sensitive** (temperature T, soft/hard-label weighting, training length), requiring tuning. **Conclusion: edge deployment brings models to devices via the compression trio — quantization is almost-free 4x compression (first choice), pruning trades sparsity for size (structured is more practical), distillation transfers big-model knowledge to small models (significantly effective only on hard tasks); but honestly, compression isn't "more complex is better" — quantization and "choosing the right small architecture" are often the simplest and most effective moves, while complex distillation pays off only when "the small model can't learn well on its own." Which to choose depends on your device constraints and task difficulty.**

> 💼 **实战视角 / Practical angle**
> **中文**:边缘部署落地:①**先量化**——最简单最划算, PTQ(训练后量化)一行搞定, 精度不够上 QAT(量化感知训练); 端侧 LLM 用 4-bit(GPTQ/AWQ);②**结构化剪枝**(删通道/头)比非结构化更易加速, 剪后微调恢复精度;③**蒸馏**用于把大模型能力压进可部署的小模型(DistilBERT/TinyBERT), 但只在难任务、学生容量不足时才显著划算;④**先考虑'选对小架构'**(MobileNet/EfficientNet 这类为边缘设计的模型), 别总想着压大模型;⑤**部署运行时**:TFLite(安卓)、Core ML(iOS)、ONNX Runtime(通用)、TensorRT(NVIDIA)、llama.cpp/GGUF(端侧 LLM), 它们做算子融合+量化推理;⑥**测真实设备指标**:延迟、内存峰值、功耗、发热, 不只看精度。面试金句:*"边缘设备内存算力有限, 靠压缩三件套:量化(fp32→int8, 4x小几乎无损, 首选)、剪枝(置零小权重, 结构化更硬件友好)、蒸馏(小模型学大模型软标签, 难任务才显著有效); 还要选对小架构(MobileNet), 用 TFLite/ONNX Runtime/Core ML 部署做量化和算子融合; 端侧 LLM 靠 4-bit 量化; 核心是体积/延迟/功耗与精度的权衡, 且量化和选对架构常比复杂蒸馏更实用。"*
> **English**: Edge deployment in practice: ① **quantize first** — simplest and most cost-effective, PTQ (post-training quantization) is one line, use QAT (quantization-aware training) if accuracy is insufficient; on-device LLMs use 4-bit (GPTQ/AWQ); ② **structured pruning** (remove channels/heads) is easier to accelerate than unstructured, fine-tune after pruning to recover accuracy; ③ **distillation** to compress big-model capability into a deployable small model (DistilBERT/TinyBERT), but significantly worthwhile only on hard tasks with insufficient student capacity; ④ **consider "choosing the right small architecture" first** (edge-designed models like MobileNet/EfficientNet), don't always think about compressing big models; ⑤ **deployment runtimes**: TFLite (Android), Core ML (iOS), ONNX Runtime (general), TensorRT (NVIDIA), llama.cpp/GGUF (on-device LLMs), doing operator fusion + quantized inference; ⑥ **measure real device metrics**: latency, peak memory, power, heat, not just accuracy. Interview line: *"Edge devices have limited memory/compute, addressed by the compression trio: quantization (fp32→int8, 4x smaller, nearly lossless, first choice), pruning (zero small weights, structured is more hardware-friendly), distillation (small model learns the big model's soft labels, significantly effective only on hard tasks); also choose the right small architecture (MobileNet), deploy with TFLite/ONNX Runtime/Core ML for quantization and operator fusion; on-device LLMs rely on 4-bit quantization; the core is trading size/latency/power against accuracy, and quantization plus the right architecture are often more practical than complex distillation."*

---
### 小结 / Summary
- **中文**:边缘部署要压缩模型(小内存/弱算力/低延迟/离线/隐私); 三件套:量化、剪枝、知识蒸馏。
- **English**: Edge deployment requires compression (small memory/weak compute/low latency/offline/privacy); the trio: quantization, pruning, knowledge distillation.
- **中文**:量化(fp32→int8)几乎免费 4x 压缩(精度 0.886→0.886, 首选); 剪枝 80% 稀疏小掉点; 蒸馏难任务才显著有效。
- **English**: Quantization (fp32→int8) is almost-free 4x compression (accuracy 0.886→0.886, first choice); pruning gives 80% sparsity at small cost; distillation is significantly effective only on hard tasks.
- **中文**:诚实:压缩非越复杂越好——量化和'选对小架构'常最实用; 部署用 TFLite/ONNX Runtime/Core ML, 端侧 LLM 用 4-bit。
- **English**: Honestly: compression isn't "more complex is better" — quantization and "the right small architecture" are often most practical; deploy with TFLite/ONNX Runtime/Core ML, on-device LLMs use 4-bit.
